# Performance plots from saved pipeline CSVs

After you run **QLattice** or **TabPFN** with `--plot-dir` (or `--export-csv-dir`), each run writes a CSV such as:

- `qlattice_qcr_labels_seed42.csv`
- `tabpfn_qcr_labels_seed42.csv`

Columns include `split`, `loading`, **`Test status`** (Unloaded / Near L.U / Loaded), `Q_meas_m3day`, `Q_cr_pred_m3day` (or `Q_gas_pred_m3day`), and `delta_*_ref_m3day` for the Turner rule.

This notebook rebuilds **per-class test accuracy** (similar spirit to `plots/new/clean_liquid_loading_plots.ipynb`) and a **grouped bar chart**. Point `EXPORT_DIR` at the folder that contains those files.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.prediction_csv_tools import summarize_exports_dir

# Folder you passed as --plot-dir or --export-csv-dir
EXPORT_DIR = Path("../plots/my_run_exports")

summary = summarize_exports_dir(EXPORT_DIR)
summary

In [ ]:
# Grouped per-class accuracy (test fold only), same layout idea as clean_liquid_loading_plots

if summary.empty:
    raise SystemExit(f"No CSV files in {EXPORT_DIR.resolve()}")

per_class = summary.set_index("file")[["acc_unloaded", "acc_near", "acc_loaded"]].rename(
    columns={
        "acc_unloaded": "Unloaded",
        "acc_near": "Near L.U",
        "acc_loaded": "Loaded",
    }
)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(per_class.shape[1])
width = min(0.8 / max(per_class.shape[0], 1), 0.2)
for i, (model, row) in enumerate(per_class.iterrows()):
    ax.bar(x + i * width, row.values, width, label=str(model)[:40], edgecolor="0.2", linewidth=0.5)
ax.set_xticks(x + width * (per_class.shape[0] - 1) / 2)
ax.set_xticklabels(per_class.columns)
ax.set_ylabel("Fraction correct (test)")
ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_title("Per-class accuracy from exported CSVs")
fig.tight_layout()
plt.show()

### Optional: CV vs test (manual)

If you saved **K-fold** exports (`*_cv_last_fold.csv`), you only have one fold’s arrays — use the printed CV summary in the terminal for CV mean ± std, and the CSV row `split==test` for that fold’s test accuracy. For a full **CV vs holdout test** bar chart like the original notebook, run a dedicated evaluation script that logs both numbers per model.